# Day 3 — Building Reliable Multi-Agent Systems

## Course overview

Today, you will learn how to design and build multi-agent systems using different orchestration patterns.

You will begin with a brief recap of a single agent that combines instructions, tools and structured outputs. From there, you will explore when a single agent becomes difficult to manage and when it makes sense to introduce specialised agents.

You will learn three main orchestration approaches:

- using a manager agent to coordinate specialists;
- using handoffs to route work to the most appropriate agent;
- using Python to control workflows where the sequence is already known.

You will also explore practical techniques for making multi-agent systems more __reliable__, including tracing, guardrails, structured outputs and evaluation.

In the afternoon, you will apply these ideas by designing, building, testing and reviewing your own multi-agent system.

## How to use this notebook

Same conventions as Day 2. Labels tell you what's core lesson, what's reference, and what you should try yourself:

| Label | Meaning |
|---|---|
| 🟢 **Core lesson** | Follow during the morning session |
| 🔍 **Predict before running** | Pause, decide what you expect, then run |
| ✅ **Check your understanding** | A short exercise to verify the concept |
| 🎯 **Stretch task** | Extra exercise if you finish early or want to push further |
| 📚 **Self-study** | Read on your own time |
| ⚠️ **Production note** | Important engineering context |

## Three parts

| Part | Topic |
|---|---|
| **Part 1** | Single-agent recap and when to reach for multi-agent |
| **Part 2** | Three orchestration patterns — Manager, Handoff, Python |
| **Part 3** | Making multi-agent systems reliable — tracing, guardrails |

## Setup

In [1]:
# loading enviroment variables

from dotenv import load_dotenv
from pathlib import Path
import os

project_root = Path.cwd()

while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent

env_path = project_root / ".env"

print(".env exists:", env_path.exists())

load_dotenv(env_path)

.env exists: True


True

In [2]:
from typing import Literal
from pydantic import BaseModel, Field
from agents import (
    Agent,
    Runner,
    SQLiteSession,
    function_tool,
    GuardrailFunctionOutput,
    InputGuardrail,
    trace,
)

MODEL = os.getenv("OPENAI_MODEL")

print(f"Using model: {MODEL}")

Using model: gpt-4o-mini


#### [OPTIONAL] Setting up Tracing

In [7]:
# ------------------------------------------------------------
# Student tracing setup
# ------------------------------------------------------------
# Leave OPENAI_API_KEY alone — it is used for model calls.
#
# Your job:
# 1. Create your own OpenAI Platform API key.
# 2. Add it to the .env file as OPENAI_TRACING_API_KEY.
# 3. Run this cell before running agent examples.
#
# Your .env file should contain BOTH keys:
#
# OPENAI_API_KEY=the_class_model_key
# OPENAI_TRACING_API_KEY=your_personal_tracing_key
#
# This means:
# - model calls use the class key
# - trace uploads use your personal key
#
# Get the API Key from here: https://platform.openai.com/api-keys








# from agents import set_tracing_export_api_key

# tracing_key = os.getenv("OPENAI_TRACING_API_KEY")

# if not tracing_key:
#     raise ValueError("OPENAI_TRACING_API_KEY is missing. "
#         "Add your own OpenAI API key to the .env file first.")

# set_tracing_export_api_key(tracing_key)

# print("Tracing key loaded successfully.")

## Introduction to Agent Orchestration

AI agents do more than generate responses. They work towards a goal by making decisions, using tools, and progressing through a sequence of steps.

The way these steps are organised and controlled is known as **agent orchestration**.

A well-designed orchestration approach helps an agent system remain:

* **Clear**, so its behaviour is easier to understand
* **Efficient**, so it does not introduce unnecessary complexity
* **Reliable**, so it can complete tasks consistently
* **Recoverable**, so it can respond appropriately when something goes wrong

In general, orchestration patterns fall [into two categories](https://openai.com/business/guides-and-resources/a-practical-guide-to-building-ai-agents/):

* **Single-agent systems**
  A single model, equipped with the appropriate tools and instructions, executes the workflow in a loop. It reasons, takes action, evaluates the result, and continues until the task is complete.

* **Multi-agent systems**
  Workflow execution is distributed across multiple coordinated agents. Each agent may have a specialised role, set of tools, or area of responsibility, with agents delegating tasks or handing control to one another.

Neither approach is automatically better. The right choice depends on the complexity of the workflow, the number of tools involved, and the level of reliability and control the system requires.

In this course, we will explore how these orchestration patterns work, when to use each one, and how to design agent systems that remain dependable as workflows become more complex.


---

# Part 1 — Single-Agent Recap and When to Reach for Multi-Agent

Before we build multi-agent systems, it's worth re-anchoring on what a single agent looks like and recognising when one isn't enough.

You've spent Day 2 building agents with tools, structured outputs, and memory. This part recaps that pattern briefly, then identifies the signals that tell you it's time to split the work across specialists.


## Recap: Single-Agent Systems

🟢 **Core lesson**

A single-agent system uses one agent to manage the complete workflow. The agent receives a request, decides what action to take, uses any available tools, and continues until it can return a final answer.

A typical single-agent system contains:

- __Instructions__ that define the agent’s role and behaviour
- __Tools__ that allow the agent to retrieve information or perform actions
- __A LLM model__ (the brains) that decides what to do next
- __An output format__ that defines how the result should be returned

```mermaid
flowchart LR
    U[User request] --> A[Single agent]
    A -->|Needs information| T[Tool]
    T --> A
    A --> O[Structured output]
```

In this short recap, ____we will build an engineering component adviser____.  
It will look up information about an electronic component and return a simple structured response.

### Example Single Agent System: Engineering Component Adviser

In [3]:
# ------------------------------------------------------------------
# Component data: We will use a small local dictionary as our data source.
# ------------------------------------------------------------------

COMPONENTS = {
    "resistor": {
        "purpose": "Limits current in a circuit.",
        "common_fault": "It may fail open after overheating.",
    },
    "capacitor": {
        "purpose": "Stores electrical charge.",
        "common_fault": "It may lose capacitance or begin leaking.",
    },
    "thermistor": {
        "purpose": "Changes resistance as temperature changes.",
        "common_fault": "It may become inaccurate after thermal damage.",
    },
}


# ------------------------------------------------------------------
# Tool: The tool allows the agent to look up one component.
# ------------------------------------------------------------------

@function_tool
def look_up_component(component: str) -> str:
    """Look up the purpose and common fault of an electronic component."""
    record = COMPONENTS.get(component.strip().lower())

    if record is None:
        return f"No information was found for {component!r}."

    return (
        f"Purpose: {record['purpose']}\n"
        f"Common fault: {record['common_fault']}"
    )


# ------------------------------------------------------------------
# Structured output: The agent will return three fields.
# ------------------------------------------------------------------

class ComponentAdvice(BaseModel):
    component: str
    purpose: str
    common_fault: str


# ------------------------------------------------------------------
# Agent
# ------------------------------------------------------------------

component_adviser = Agent(
    name="Component Adviser",
    model=MODEL,
    instructions=(
        "You help engineering students understand electronic components. "
        "Always use the component lookup tool to find approved information. "
        "Return the component name, purpose, and common fault. "
        "Do not invent information when a component is unavailable. Just say you don't know what it is."
    ),
    tools=[look_up_component],
    output_type=ComponentAdvice,
)


In [4]:
# Run the agent


# Known component
user_query = "What does a resistor do, and what is one common fault?"

result = await Runner.run(
    component_adviser,
    user_query,
)

advice = result.final_output
print(advice.model_dump_json(indent=2))

{
  "component": "resistor",
  "purpose": "Limits current in a circuit.",
  "common_fault": "It may fail open after overheating."
}


In [4]:

# Unknown component
user_query = "What does a radio do, and what is one common fault?"

result = await Runner.run(
    component_adviser,
    user_query,
)

advice = result.final_output
print(advice.model_dump_json(indent=2))

{
  "component": "radio",
  "purpose": "I don't know what it is.",
  "common_fault": "I don't know what it is."
}


This example brings together the main parts of a single-agent system:

| Component | Implementation |
|---|---|
| **Instructions** | The role and behavioural rules |
| **Tool** | `look_up_component()` |
| **Model** | `MODEL` |
| **Structured output** | `ComponentAdvice` |
| **Execution Loop** | `Runner.run()` |

### ✅ Check your understanding

1. What information in this example is provided by the tool rather than the model?
2. What would happen if the user asked about a component that is not in the dictionary?
3. What advantage does `ComponentAdvice` provide over returning only free-form text?
4. Add a `power_rating_watts` field to `ComponentAdvice`. Update the look-up tool to return this information for at least one component. What other changes do you need to make so the structured output remains valid?

### When Does One Agent Become Difficult to Manage?

For the current example, a single-agent design is appropriate. 
The agent has a narrow responsibility, clear instructions and one well-defined tool, making it easy to understand, test and control.

Now imagine expanding it so that it must also:
- diagnose electrical, mechanical and sensor faults;
- search several technical manuals;
- recommend maintenance actions;
- apply different safety rules;
- decide when to contact a human engineer;
- choose between similar tools.

As these responsibilities grow, the prompt may become dominated by conditional logic, while tools become harder to distinguish and select correctly. 
This can reduce reliability and make the system more difficult to maintain.

A multi-agent design may be useful when improving the prompt and tool definitions no longer produces reliable behaviour.

__[Two common warning signs are](https://openai.com/business/guides-and-resources/a-practical-guide-to-building-ai-agents/):__

- __Complex decision logic:__ The agent must follow many conditional branches that are difficult to extend or test.
- __Tool overload:__ The agent struggles to choose between tools with similar or overlapping purposes.

__In these situations, the workflow can be divided between specialised agents, each with a smaller and more clearly defined responsibility.__

This leads to the next design approach: the __Multi-Agent System.__


---

# Part 2 — Multi-Agent Orchestration

When one agent isn't enough, the question becomes: **how should multiple agents work together?**

This part covers the three patterns that show up in most production multi-agent systems. Same example domain throughout where it helps — same specialists, different orchestrations — so you can see how the pattern choice changes the shape of the system.

## Multi-Agent Systems

🟢 **Core lesson**

Once a single agent becomes difficult to manage, responsibilities can be divided across specialised agents.  

There are three common patterns to coordinate this work across multiple agents:

| Patterns             | Main idea                                              | Best suited to                    |
|----------------------|--------------------------------------------------------|-----------------------------------|
| Manager              | One agent calls specialists and combines their results | Delegation and synthesis          |
| Handoffs             | One agent transfers control to another                 | Routing and specialist ownership  |
| Python orchestration | Code explicitly controls the sequence                  | Predictable, fixed workflows      |

### Pattern: Manager (Agents as Tools)

A central agent remains in control, delegates bounded tasks to specialist agents by using them as tools, gathers their outputs, and synthesises the final response for the user.


```mermaid
flowchart TD
    U[User] --> M[Manager]
    M --> S1[Specialist A]
    M --> S2[Specialist B]
    M --> S3[Specialist C]
    S1 --> M
    S2 --> M
    S3 --> M
    M --> U
```

Use the manager pattern when:

- specialists handle clearly bounded subtasks;
- the final response must combine outputs from several specialists;
- one central agent should retain control of the workflow, user interaction, and synthesis.

### Pattern: Decentralised(Handoff)

A peer agent transfers control and the current conversation state to a specialist agent, which takes over execution and interacts directly with the user without central control or synthesis.

```mermaid
flowchart LR
    U[User] --> T[Triage agent]
    T -->|IT request| I[IT specialist]
    T -->|Academic request| A[Academic specialist]
    T -->|Records request| R[Student records specialist]
```

Use the decentralised handoff pattern when:

* multiple agents operate as peers rather than under a central manager;
* an agent should transfer workflow control to another specialist;
* the latest conversation state must move with the handoff;
* the receiving agent should continue execution and interact directly with the user;
* no single agent needs to maintain central control or synthesise every result.


### Pattern: Python (or code controlled) orchestration 

Python (Code) explicitly controls a known sequence.

```mermaid
flowchart LR
    P[Python controller] --> A[Step 1 agent]
    A --> B[Step 2 agent]
    B --> C[Step 3 agent]
    C --> O[Output]
```

---
## Example Implementations

### A. Manager Pattern

#### Example 1: Translation agent

**Specialists:**

- Spanish translation agent;
- French translation agent;
- Italian translation agent.

**Example request:**
> Translate “hello” into Spanish, French and Italian.

The manager calls each requested language specialist and combines the translations into one response.

In [6]:
# Note: Example taken from https://openai.com/business/guides-and-resources/a-practical-guide-to-building-ai-agents/

from agents import Agent, Runner


# Specialists agents
# ----------------------------------------

spanish_agent = Agent(
    name="Spanish Translator",
    model=MODEL,
    instructions=(
        "Translate supplied English text into natural Spanish. "
        "Return only the translation."
    ),
)

french_agent = Agent(
    name="French Translator",
    model=MODEL,
    instructions=(
        "Translate supplied English text into natural French. "
        "Return only the translation."
    ),
)

italian_agent = Agent(
    name="Italian Translator",
    model=MODEL,
    instructions=(
        "Translate supplied English text into natural Italian. "
        "Return only the translation."
    ),
)



# Expose agents as tools
# ----------------------------------------
translation_manager = Agent(
    name="Translation Manager",
    model=MODEL,
    instructions=(
        "Manage multilingual translation requests. "
        "Call the relevant translation specialists. "
        "Keep ownership of the final response. "
        "Label each translation clearly and do not translate languages "
        "that the user did not request."
    ),
    tools=[
        spanish_agent.as_tool(
            tool_name="translate_to_spanish",
            tool_description="Translate English text into Spanish.",
        ),
        french_agent.as_tool(
            tool_name="translate_to_french",
            tool_description="Translate English text into French.",
        ),
        italian_agent.as_tool(
            tool_name="translate_to_italian",
            tool_description="Translate English text into Italian.",
        ),
    ],
)

In [7]:
# Run and inspect
with trace("course_translation_manager"):
    result = await Runner.run(
        translation_manager,
        "Translate 'The laboratory opens at nine' into Spanish,French and Italian."
    )

print(result.final_output)

Here are the translations for "The laboratory opens at nine":

- **Spanish:** El laboratorio abre a las nueve.
- **French:** Le laboratoire ouvre à neuf heures.
- **Italian:** Il laboratorio apre alle nove.


#### ✅ Check your understanding

Add a German translation specialist alongside Spanish, French, and Italian. Test the manager with: *"Translate 'hello' into German and French."* Check that it calls only the two specialists requested, not all four.

#### Example 2: Social media post creator

**Specialists:**

* research agent;
* LinkedIn agent.

**Example request:**

> Create a LinkedIn post about a current trend in AI agents.

The manager asks the research agent to find three current facts, then passes them to the LinkedIn agent and returns the completed campaign as structured output.


In [8]:

from pydantic import BaseModel
from IPython.display import Markdown, display
from agents import Agent, Runner, WebSearchTool



# Structured output: The manager returns the completed campaign.
class SocialMediaCampaign(BaseModel):
    topic: str
    key_facts: list[str]
    linkedin_post: str


# Specialists agents
# ----------------------------------------

# Specialist 1: Researches a current topic using web search.

research_agent = Agent(
    name="Research Specialist",
    model=MODEL,
    instructions=(
        "Research the requested topic using web search. "
        "Return exactly three concise, factual bullet points in Markdown. "
        "Do not include unnecessary background information."
    ),
    tools=[WebSearchTool()],
)


# Specialist 2: Turns research into a LinkedIn post.

linkedin_agent = Agent(
    name="LinkedIn Specialist",
    model=MODEL,
    instructions=(
        "You are an experienced LinkedIn content specialist. "
        "Write a concise LinkedIn post using the research provided. "
        "Use clear Markdown, a professional tone, and no more than 120 words. "
        "Include a short heading and up to three relevant hashtags."
    ),
)


# ------------------------------------------------------------------
# Manager: Calls the specialists agents (as tools) and synthesises their outputs.
# ------------------------------------------------------------------

manager_agent = Agent(
    name="Social Media Manager",
    model=MODEL,
    instructions=(
        "Create a short LinkedIn campaign about the user's topic. "
        "First call the research specialist to obtain exactly three current facts. "
        "Then pass those facts to the LinkedIn specialist. "
        "Return the topic, the three key facts, and the finished LinkedIn post. "
        "Do not perform the specialists' tasks yourself."
    ),
    tools=[
        research_agent.as_tool(
            tool_name="research_topic",
            tool_description="Find three current facts about a topic.",
        ),
        linkedin_agent.as_tool(
            tool_name="write_linkedin_post",
            tool_description=(
                "Turn supplied research into a concise Markdown LinkedIn post."
            ),
        ),
    ],
    output_type=SocialMediaCampaign,
)

In [9]:
# Run the manager.
user_query = "Create a LinkedIn post about a current trend in AI agents."

# Run and inspect
with trace("social_media_manager"):
    result = await Runner.run(
        manager_agent,
        user_query,
    )

campaign = result.final_output

print(campaign.topic)

Current Trends in AI Agents


In [13]:
# Display the structured output as formatted Markdown.
display(Markdown(f"# {campaign.topic}"))
display(Markdown(campaign.linkedin_post))


# Current Trends in AI Agents

### Embracing the Future of AI Agents

The landscape of AI agents is evolving rapidly:

1. **From Experimentation to Deployment**: Enterprises are moving towards full-scale adoption, necessitating new governance frameworks.
   
2. **Heightened Security Risks**: Increased autonomy calls for robust zero trust architectures and advanced monitoring solutions.
   
3. **Complex Digital Workforces**: AI agents are transcending simple tasks, now capable of coordinating workflows and anticipating organizational needs.

As we navigate these changes, the focus on governance and security will be paramount. 

#ArtificialIntelligence #AIAgents #DigitalTransformation

#### 🎯 Stretch task

Add a *Twitter Specialist* alongside the LinkedIn agent. Update the manager and the structured output so the campaign contains both posts (LinkedIn and Twitter). What does the structured output need to look like to support both?

*Hint: think about whether each platform's post is a separate field, or whether they live inside a list.*

### B. Handoff Pattern

#### Example 1: Customer support router

__Specialists:__

- Order-status agent;
- Refund agent;
- General support agent.

__Example request:__

>I want a refund for an item I purchased.

The triage agent identifies the request and hands control, along with the conversation context, to the refund agent. The refund agent then continues the conversation directly with the user.

Why it works well:
It clearly demonstrates routing, transfer of control and specialist ownership without a central agent synthesising the final response.


```mermaid
flowchart LR
    U[User] --> T[Triage Agent]

    T -->|Order or delivery| O[Order Status Agent]
    T -->|Refund or return| R[Refund Agent]
    T -->|General support| G[General Support Agent]

    O -.->|Responds directly| U
    R -.->|Responds directly| U
    G -.->|Responds directly| U
```





In [15]:
from agents import Agent, Runner, trace

# ------------------------------------------------------------------
# Specialist agents
# ------------------------------------------------------------------

order_status_agent = Agent(
    name="OrderStatus",
    model=MODEL,
    handoff_description="Handles questions about order tracking and delivery.",
    instructions=(
        "Help the user with order-status and delivery questions. "
        "Explain that this is a demonstration and do not invent real order data."
    ),
)

refund_agent = Agent(
    name="Refund",
    model=MODEL,
    handoff_description="Handles refund and return requests.",
    instructions=(
        "Help the user understand the refund or return process. "
        "Ask for any essential missing information. "
        "Do not claim that a refund has actually been processed."
    ),
)

general_support_agent = Agent(
    name="GeneralSupport",
    model=MODEL,
    handoff_description="Handles general customer-support questions.",
    instructions=(
        "Answer general customer-support questions clearly and concisely."
    ),
)


# ------------------------------------------------------------------
# Triage agent: Selects a specialist and hands over control.
# ------------------------------------------------------------------

triage_agent = Agent(
    name="Triage",
    model=MODEL,
    instructions=(
        "Identify what kind of support the user needs and hand off to "
        "the most suitable specialist. Do not answer the support request yourself."
    ),
    handoffs=[
        order_status_agent,
        refund_agent,
        general_support_agent,
    ],
)

In [16]:
# Run and inspect
customer_requests = [
    "I bought the wrong item and would like a refund.",
     "I would like to know about the store opening hours in London.",
]

for request in customer_requests:
    with trace("handoff_customer_support_router"):
        result = await Runner.run(triage_agent, request)

    print("\n\n---\n")
    print("REQUEST:", request)
    print("FINAL AGENT:", result.last_agent.name)
    print("RESPONSE:", result.final_output)



---

REQUEST: I bought the wrong item and would like a refund.
FINAL AGENT: Refund
RESPONSE: I can help you with the refund process. To assist you better, could you please provide the following information?

1. The item you purchased and its order number.
2. When did you buy the item?
3. Did you already initiate the return process, or is this your first request?
4. Are there any special conditions or policies regarding returns for the item?

Once I have this information, I can guide you through the next steps!


---

REQUEST: I would like to know about the store opening hours in London.
FINAL AGENT: GeneralSupport
RESPONSE: Store opening hours in London can vary widely by location and type of store. Typically, many shops in central London open around 9:00 AM and close at 6:00 PM on weekdays. Some may extend their hours until 8:00 PM or later. On weekends, hours may be shorter, often around 10:00 AM to 5:00 PM. It's best to check specific store websites or call ahead for accurate hour

##### What to observe

- The triage agent should transfer control.
- The selected specialist produces the final response.
- `result.last_agent` shows which agent ended the run.
- The trace should contain a handoff event.

#### ✅ Check your understanding

Add a fourth specialist for **billing questions** (e.g. duplicate charges, invoice queries). Update the triage agent's instructions so it knows when to hand off to billing. Test it with: *"I was charged twice for my last order — can someone look into it?"*

Then open the trace and verify which agent ended up handling the request.

--
### C. Code Controlled Pattern

#### Example 1: Blog post pipeline

__Agents:__

- outline agent;
- writing agent;
- editing agent.

__Example request:__

> Write a short blog post explaining the benefits of remote working.

__A linear workflow should not automatically be turned into a chain of handoffs.__  
Here the order is known. Python runs the agents in a fixed sequence:
- create an outline;
- write the draft;
- edit the draft.

__Why it works well:__
The sequence is obvious, the outputs can remain short, and no agent needs to decide what happens next.

```mermaid
flowchart LR
    U[User topic] --> P[Python controller]
    P --> O[Outline Agent]
    O --> W[Writing Agent]
    W --> E[Editing Agent]
    E --> F[Final blog post]
```

In [17]:
import json

from pydantic import BaseModel
from agents import Agent, Runner, trace
from IPython.display import Markdown, display

# ------------------------------------------------------------------
# Structured outputs
# ------------------------------------------------------------------

class BlogOutline(BaseModel):
    title: str
    sections: list[str]


class BlogDraft(BaseModel):
    title: str
    content: str


class FinalBlogPost(BaseModel):
    title: str
    content: str


# ------------------------------------------------------------------
# Agents
# ------------------------------------------------------------------

outline_agent = Agent(
    name="Outline Agent",
    model=MODEL,
    instructions=(
        "Create a concise outline for a short blog post. "
        "Return a title and exactly three section headings."
    ),
    output_type=BlogOutline,
)

writing_agent = Agent(
    name="Writing Agent",
    model=MODEL,
    instructions=(
        "Write a short blog post using only the supplied topic and outline. "
        "Use Markdown and keep the post below 250 words."
    ),
    output_type=BlogDraft,
)

editing_agent = Agent(
    name="Editing Agent",
    model=MODEL,
    instructions=(
        "Edit the supplied blog post for clarity, flow and grammar. "
        "Preserve the meaning and return the finished post in Markdown."
    ),
    output_type=FinalBlogPost,
)


# ------------------------------------------------------------------
# Python controller: Runs the agents in a fixed sequence.
# ------------------------------------------------------------------

async def create_blog_post(topic: str) -> FinalBlogPost:
    outline_result = await Runner.run(
        outline_agent,
        topic,
    )
    outline = outline_result.final_output

    writing_input = {
        "topic": topic,
        "outline": outline.model_dump(),
    }

    draft_result = await Runner.run(
        writing_agent,
        json.dumps(writing_input, indent=2),
    )
    draft = draft_result.final_output

    editing_input = {
        "topic": topic,
        "draft": draft.model_dump(),
    }

    final_result = await Runner.run(
        editing_agent,
        json.dumps(editing_input, indent=2),
    )

    return final_result.final_output





In [18]:
# Run pipeline with trace

topic = "The benefits of remote working"

with trace("blog_post_cc_pipeline"):
    blog_post = await create_blog_post(topic)

In [19]:
# Display results
print(blog_post.model_dump_json(indent=2))

#display(Markdown(f"{blog_post.content}"))

{
  "title": "The Benefits of Remote Working",
  "content": "# The Benefits of Remote Working\n\nAs remote working becomes the norm in today’s professional landscape, it’s essential to recognize its myriad benefits.\n\n## Increased Flexibility and Work-Life Balance\nOne of the most significant advantages of remote work is the flexibility it offers. Employees can tailor their work schedules to fit personal commitments, leading to an improved work-life balance. This flexibility can result in higher job satisfaction and reduced stress, ultimately enhancing productivity.\n\n## Cost Savings for Employers and Employees\nBoth employers and employees can experience substantial cost savings through remote work arrangements. Employers save on overhead costs like office space and utilities, while employees cut down on commuting expenses, meals, and work attire. This financial relief can significantly improve quality of life and maintain motivation.\n\n## Access to a Broader Talent Pool\nRemote wo

##### __Why this is not a handoff chain?__

The sequence is predetermined. A language model does not need to decide that order the agents. Python makes the order:

- easier to test;
- easier to retry;
- easier to monitor;
- less likely to skip a stage.

#### 🎯 Stretch task

Add a fourth stage to the pipeline: a **SEO Specialist** that suggests three relevant keywords for the post. Where in the Python controller does this stage belong? Should it run before, after, or alongside the editor?

*Hint: there's more than one defensible answer — pick one and justify it in a comment.*

---

# Part 3 — Making Multi-Agent Systems Reliable

Choosing an orchestration pattern determines how work moves through the system, but it does not guarantee correct behaviour. A multi-agent system may still call the wrong specialist, skip a stage, or produce an unsupported answer.

This part covers two practices that make systems more reliable: **tracing** (so you can see what happened) and **guardrails** (so you can constrain what's allowed to happen). Evaluation — the third reliability practice — gets fuller treatment on Day 4, where it's central to building RAG systems responsibly.

## Making Agent Systems Reliable

Choosing an orchestration pattern determines how work moves through the system, but it does not guarantee that the system will behave correctly.

A multi-agent system may still:

* call the wrong specialist;
* use an unnecessary tool;
* skip an important stage;
* repeat the same action;
* produce an unsupported answer;
* fail without making the cause clear.

__To improve reliability__, we need ways to ___observe___, ___constrain___ and ___test___ the workflow.

In this section, we will focus on three practices:

1. **Tracing** — understanding what happened during a run.
2. **Guardrails** — checking inputs, outputs or actions against defined rules.
3. **Evaluation** — testing whether the system behaves as expected across representative cases.


```mermaid
flowchart LR
    O[Orchestrated agent system] --> T[Tracing]
    O --> G[Guardrails]
    O --> E[Evaluation]

    T --> R[Understand behaviour]
    G --> S[Enforce boundaries]
    E --> Q[Measure quality and reliability]
```

These practices answer different questions:

| Practice   | Main question                                            |
| ---------- | -------------------------------------------------------- |
| Tracing    | What happened during the run?                            |
| Guardrails | Should this input, output or action be allowed?          |
| Evaluation | Does the system behave correctly across different cases? |

## Tracing

🟢 **Core lesson**

The Agents SDK includes tracing for model generations, tool calls, handoffs, guardrails and related events.  
Traces help answer questions such as:

- Which agent handled the request?
- Which tools were called?
- In what order?
- Where did latency occur?
- Did a guardrail run?
- Did the agent loop unexpectedly?
- Did the system make more calls than expected?

OpenAI Traces: https://platform.openai.com/logs?api=traces

LEARN MORE: https://openai.github.io/openai-agents-python/tracing/

### Name traces meaningfully

You should be able to decifer a particular run based on the trace name. 
Here are some practical  tips for:
- Avoid generic names such as `test` or `agent run`.
- Come up with a naming convetion for your project or team.
- And feel free to parameterise it. 
 - Here is an example:
    -  `trace_name = coursework_{workflow_name}_{user_name}`


### Reading a Trace

__The structure of an agent trace__

Traces in the OpenAI Agents SDK have three levels of nesting that you need to recognise:


```
Trace                  ← top-level container (what you named via trace("sacial_media_manager"))
└── Task               ← one run of Runner.run()
    └── Agent          ← which agent is running (Social Media Manager)
        └── Turn       ← one cycle of "ask the model, do what it says"
            ├── POST /v1/responses    ← the actual model call
            └── tool calls            ← what the model decided to do
```

<br>
<br>

__Let's do an example__

In [20]:
# Let's do the social media manager again

# example of parameterise naming 
run_number = 2
trace_name = f"social_media_manager_run_{run_number}"


user_query = "Create a LinkedIn post about a current trend in AI agents."
with trace(trace_name):
    result = await Runner.run(
        manager_agent,
        user_query,
    )


# campaign = result.final_output
# print(campaign.topic)

Read the trace at: https://platform.openai.com/logs?api=traces

### ✅ Check your understanding

Can be done during Lab session

For one manager run and one handoff run, record:


| Observation                       | Manager | Handoff |
|-----------------------------------|---------|---------|
| Number of agents involved         |         |         |
| Number of tool calls              |         |         |
| Number of handoffs                |         |         |
| Final agent                       |         |         |
| Was the expected specialist used? |         |         |
| Any unnecessary calls?            |         |         |

## Guardrails

🟢 **Core lesson**

Agents can make decisions, call tools and interact with external systems.  
This makes them useful, but it also means we should check what enters and leaves an agent.

A __guardrail__ is a validation check placed around an agent.

Guardrails can:

- Reject requests that are outside the agent's purpose
- Detect unsafe or sensitive content
- Prevent an inappropriate response from reaching the user
- Stop expensive agent runs early
- Add application-specific rules


> You can think of a guardrail as a safety checkpoint. It does not perform the main task; it decides whether the task should be allowed to continue.


#### Types of guardrails

The Agents SDK supports guardrails at different points in a workflow:

- __Input guardrails__ check the user's initial request.
- __Output guardrails__ check the agent's final answer.
- __Tool guardrails__ check information before or after a function tool runs.

```mermaid
flowchart LR
    A[User request] --> B{Input guardrail}
    B -->|Allowed| C[Agent]
    B -->|Blocked| D[Stop the run]
    C --> E{Output guardrail}
    E -->|Allowed| F[Return response]
    E -->|Blocked| G[Do not return response]
```

### ✅ Walk through: input guardrail for a customer-support agent

In this short exercise, we will create an input guardrail for a customer-support agent.

The agent is allowed to help with:
- Orders
- Deliveries
- Returns
- Refunds
- Payments
- Account problems

Requests about unrelated topics will be blocked.

__1. Setup: import the required classes__

In [24]:
from pydantic import BaseModel, Field

from agents import (
    Agent,
    GuardrailFunctionOutput,
    InputGuardrailTripwireTriggered,
    RunContextWrapper,
    Runner,
    TResponseInputItem,
    input_guardrail,
)

__2. Define the guardrail result__

The guardrail needs to decide whether the user’s request is related to customer support.

We will use a Pydantic model to define its structured result:

In [25]:
class SupportScopeCheck(BaseModel):
     is_customer_support_request: bool
     reasoning: str = Field( description="One sentence explaining the classification." )

The guardrail will return:
- `is_customer_support_request`: whether the request is allowed
- `reasoning`: a short explanation of the decision

For example:
- `is_customer_support_reques`t = `True`
- `reasoning` = `"The user is asking about a delayed delivery."`

__3. Create the guardrail agent__

We will use a small classification agent to check the scope of the request.

In [26]:
scope_guardrail_agent = Agent(
    name="Support Scope Classifier",
    instructions=(
        "Decide whether the user's request is genuinely about customer support. "
        "Allowed topics include orders, deliveries, returns, refunds, payments "
        "and account problems. "
        "Requests about entertainment, shopping recommendations, general "
        "knowledge or unrelated writing tasks are outside scope."
    ),
    output_type=SupportScopeCheck,
)

This agent does not answer the customer’s question.

Its only job is to classify the request.

__4. Create the input guardrail__

Use the `@input_guardrail` decorator to turn an asynchronous Python function into a guardrail.

Please note that the The decorator above uses `run_in_parallel=False`. It's set to true by default.  
This turns in into a blocking input guardrail. 
Blocking matters because input guardrails may otherwise run in parallel with the main agent.   
Setting it to `False` ensures the input check completes before the protected agent begins.

In [27]:
@input_guardrail(name="customer_support_scope",
run_in_parallel=False,  # Complete the safety check before the main agent starts.
)
async def customer_support_scope(
    ctx: RunContextWrapper[None], agent: Agent, input_data: str | list[TResponseInputItem],
) -> GuardrailFunctionOutput:
    result = await Runner.run(
        scope_guardrail_agent,
        input_data,
        context=ctx.context,
    )

    check = result.final_output

    return GuardrailFunctionOutput(
        output_info=check,
        tripwire_triggered=not check.is_customer_support_request,
    )

The most important line is: `tripwire_triggered=not check.is_customer_support_request`. 

This will trigger the guardrail when the request is not related to customer support.

__5. Finally Attach the guardrail to the main agent__

Create the customer-support agent and attach the guardrail:

In [28]:
support_agent = Agent(
    name="Customer Support Triage Agent",
    instructions=(
        "Help customers with orders, deliveries, returns, refunds, "
        "payments and account problems. "
        "Keep your answers short, clear and helpful."
    ),
    input_guardrails=[customer_support_scope],
)

The guardrail is attached using:

`input_guardrails=[customer_support_scope]`

A guardrail will not run unless it is attached to an agent.

__6. Test it yourself__


In [30]:
test_requests = [
    "I was charged twice for my order.",  # Allowed
    #"I have forgotten my account password.",  # Allowed
    "Recommend a film for tonight.",  # Not allowed
    "Explain how photosynthesis works.",  # Not allowed
]

for request in test_requests:
    
    print("-"* 10)
    print(f"User: {request}")

    try:
        result = await Runner.run(support_agent,request)
        print(f"Agent: {result.final_output}")

    except InputGuardrailTripwireTriggered:
        print(
            "Blocked: this assistant can only help with "
            "customer-support enquiries."
        )

----------
User: I was charged twice for my order.
Agent: Sorry about that. I can help.

Please send:
- your order number
- the last 4 digits of the card used
- the date and amount of both charges

If you can, also upload a screenshot of the duplicate charges.
----------
User: Recommend a film for tonight.
Blocked: this assistant can only help with customer-support enquiries.
----------
User: Explain how photosynthesis works.
Blocked: this assistant can only help with customer-support enquiries.


#### ✅ Check your understanding

Modify the customer-support guardrail so questions about **gift cards** are also allowed (they're a legitimate support topic). Add a test request — *"How do I redeem a gift card?"* — and verify it now passes the guardrail.

### Some Thoughts

#### Guardrails do not have to use another agent

In this example, the guardrail uses an agent because it needs to understand the meaning of the user's request.

For example:

> The courier says my package was delivered, but I cannot find it.

The user does not say “customer support”, but the request is clearly a support enquiry.

For simple rules, you can use normal Python inside the guardrail function:

```python
if refund_amount > 100:
    raise ValueError("Refunds over £100 require approval.")
```

A useful rule is:
* Use Python for simple, deterministic checks.
* Use an agent when the check requires understanding meaning or intent.



#### Important limitations
* Input guardrails inspect the initial input and apply when their agent is the first agent in the workflow.
* Output guardrails inspect the final response produced by the final output agent.
* Tool guardrails should be used when individual function-tool calls require their own validation.



#### Further reading

* [OpenAI Agents SDK: Guardrails](https://openai.github.io/openai-agents-python/guardrails/)
* [A practical guide to building AI agents](https://openai.com/business/guides-and-resources/a-practical-guide-to-building-ai-agents/)


--
## A note on evaluation

Tracing and guardrails are two of the three core reliability practices.  
The third is **evaluation** — systematically testing whether your system behaves correctly across representative cases.

Evaluation matters most when your system has *measurable correctness* — a question with an expected answer, a retrieval with expected sources, a classification with a ground truth label. For pure-conversational multi-agent systems, "correctness" is fuzzier and harder to score automatically.

That's why we cover evaluation properly on **Day 4**, where it's central to building RAG systems responsibly. There you'll see:

- LLM-as-judge for scoring subjective qualities (tone, groundedness, relevance)
- Retrieval metrics (Hit@k, reciprocal rank)
- Citation validation against source documents
- Building a small "golden" test suite that runs on every change

The patterns transfer to multi-agent systems too — if you build a Manager-pattern translator and want to verify it produces correct translations across 20 test phrases, the LLM-as-judge approach from Day 4 is what you'd reach for.

For today, the takeaway is simpler: **multi-agent systems still need evaluation, and the practices you'll learn tomorrow apply here too.**

📚 **Further reading**

- [OpenAI evals framework](https://github.com/openai/evals) — production-grade evaluation infrastructure
- [Anthropic — Create empirical evals](https://docs.claude.com/en/docs/test-and-evaluate/develop-tests) — opinionated guide
- [LangChain — Evaluation concepts](https://docs.smith.langchain.com/evaluation/concepts) — clean overview

---

## From notebook to your codebase

Unlike Day 2 and Day 4, Day 3 doesn't ship its own `src/` modules. That's deliberate.

Day 2 taught **tool design** — a concrete craft with a clear refactor from "messy notebook code" to "production module." You can open `src/agent_workshop/` and see exactly what good Python structure for tools looks like.

Day 3 teaches **orchestration patterns** — engineering decisions about how agents work together. The lesson is in choosing the right pattern, not in code structure. The notebook code *is* the production version.

For the afternoon lab, you'll extend your existing **`agent_workshop`** package from Day 2 into a multi-agent system. The patterns you've learned today apply directly:

- Wrap your existing tools into specialist agents
- Use the **Manager pattern** when you want central control
- Use **Handoffs** when you want routing
- Use **Python orchestration** when the sequence is predictable

You don't need new infrastructure to do multi-agent — you compose what you already have.

🎯 **Stretch task — if you're comfortable with Python**

After the lab, try refactoring your multi-agent system into a proper package — `src/your_workshop/specialists.py`, `src/your_workshop/orchestration.py`, etc. Use Day 2's `agent_workshop` as the template. This is genuinely useful production practice — most multi-agent systems live in proper modules, not in notebooks.

---

## Recap

Six ideas to take into the lab:

1. **Single-agent first.** Reach for multi-agent only when one agent visibly breaks down — long instructions, too many tools, compounding failures.
2. **Three patterns cover most needs.** Manager for synthesis, Handoff for routing, Python orchestration for predictable sequences.
3. **Same problem, different shapes.** The translation system, the social media system, the support router — each illustrates a pattern. Choose based on the shape of your problem.
4. **Tracing is non-optional** in multi-agent systems. Without it, you're debugging blind.
5. **Guardrails are quality control, not security.** They constrain behaviour, but a determined attacker can sometimes bypass them. Real safety needs auth, rate limiting, and content moderation at the platform level.
6. **Multi-agent doesn't need new infrastructure.** You compose the tools and agents you already have.

## Afternoon lab brief

Build a **multi-agent system using at least one of today's orchestration patterns**. Extend your existing Day 2 agents(and reuse the tools) or start fresh — your choice.

**Requirements:**

- At least **two specialist agents** with clear, distinct roles
- Uses the chosen orchestration pattern (Manager, Handoff, or Python) — and **justify the choice in a comment**
- Uses at least **one tool** from your Day 2 work (web search, crypto, weather, etc.)
- Produces a **real artefact** — a markdown file, a structured output, something you can show

- **Optional**: Has **at least one guardrail** — input or output, your choice
- **Optional**: Wrapped in `trace(...)` so you can inspect it after running

---

**Three project suggestions** (or invent your own):

- **Trip planner** (Manager pattern) — flight finder + hotel finder + activities specialists, synthesised into a structured itinerary saved as a markdown trip brief
- **University helpdesk** (Handoff pattern) — triage agent routes between academic advisor, IT support, library, and student records specialists
- **Research report builder** (Python orchestration) — outline → research → draft → review pipeline that produces a markdown report

**If you finish early:** add a second guardrail.

This is also the system you may want to extend or use as inspiration for **Day 5's capstone**. Build something you'd actually want to use.

See you in the lab.